# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
fatal: unable to access 'https://github.com/Lv1g1/RecSys-Challenge-2025.git/': Could not resolve host: github.com


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender

optimizer = ModelOptimizer("SLIMElasticNet")

STUDY_NAME = MultiThreadSLIM_SLIMElasticNetRecommender.RECOMMENDER_NAME

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "l1_ratio": optuna_trial.suggest_float("l1_ratio", 0.4, 0.8),
        "alpha": optuna_trial.suggest_float("alpha", 1e-6, 1e2, log=True),
        "positive_only": False,
        "topK": optuna_trial.suggest_int("topK", 10, 1000, step=10)
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train)
        recommender_instance.fit(**params, workers=8)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=0
)

[I 2025-11-21 11:26:48,765] Using an existing study with name 'SLIMElasticNetRecommender' instead of creating a new one.



Study statistics: 
  Number of finished trials:  76
  Number of pruned trials:  0
  Number of complete trials:  75

Best Value: 0.28851595520973206
Best Params: {'l1_ratio': 0.42825676292409526, 'alpha': 0.0010061033871087777, 'topK': 670}


In [11]:
optuna.visualization.plot_optimization_history(optuna_study)

In [12]:
optuna.visualization.plot_param_importances(optuna_study)

In [13]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [27]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "l1_ratio": optuna_trial.suggest_float("l1_ratio", 0.3, 0.7),
        "alpha": optuna_trial.suggest_float("alpha", 1e-4, 1e-2),
        "positive_only": False,
        "topK": optuna_trial.suggest_int("topK", 600, 700)
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train)
        recommender_instance.fit(**params, workers=8, verbose=False)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [29]:
optuna_study = optimizer.create_study(
    study_name=STUDY_NAME+"_refined_1"
)

[I 2025-11-21 11:59:41,013] A new study created in RDB with name: SLIMElasticNetRecommender_refined_1


In [30]:
# Best known parameters
optuna_study.enqueue_trial({"l1_ratio": 0.42825676292409526, "alpha": 0.0010061033871087777, "topK": 670})
optuna_study.enqueue_trial({"l1_ratio": 0.6257826858334088, "alpha": 0.000818987106299432, "topK": 646})

In [31]:
optuna_study.optimize(
    refined_objective,
    n_trials=20,
    show_progress_bar=True
)

  0%|          | 0/20 [00:00<?, ?it/s]

  Fold 1/5 - Score: 0.28825247287750244
  Fold 2/5 - Score: 0.28775325417518616
  Fold 3/5 - Score: 0.28918033838272095
  Fold 4/5 - Score: 0.28783610463142395


  Fold 5/5 - Score: 0.28955358266830444
[I 2025-11-21 12:07:44,607] Trial 0 finished with value: 0.2885151505470276 and parameters: {'l1_ratio': 0.42825676292409526, 'alpha': 0.0010061033871087777, 'topK': 670}. Best is trial 0 with value: 0.2885151505470276.
  Fold 1/5 - Score: 0.28792616724967957
  Fold 2/5 - Score: 0.28761136531829834
  Fold 3/5 - Score: 0.2888050079345703


  Fold 4/5 - Score: 0.2875532805919647
[I 2025-11-21 12:13:38,377] Trial 1 finished with value: 0.28797397017478943 and parameters: {'l1_ratio': 0.6257826858334088, 'alpha': 0.000818987106299432, 'topK': 646}. Best is trial 0 with value: 0.2885151505470276.
  Fold 1/5 - Score: 0.2727256417274475
  Fold 2/5 - Score: 0.27307769656181335
  Fold 3/5 - Score: 0.27409830689430237


  Fold 4/5 - Score: 0.2730615437030792
[I 2025-11-21 12:17:09,891] Trial 2 finished with value: 0.2732407748699188 and parameters: {'l1_ratio': 0.30775802101766037, 'alpha': 0.009012095891107408, 'topK': 695}. Best is trial 0 with value: 0.2885151505470276.
  Fold 1/5 - Score: 0.26377469301223755
  Fold 2/5 - Score: 0.2636953294277191
  Fold 3/5 - Score: 0.2645294666290283


...

  Fold 4/5 - Score: 0.28716719150543213
[I 2025-11-21 13:21:08,123] Trial 15 finished with value: 0.2876375615596771 and parameters: {'l1_ratio': 0.4532273980674932, 'alpha': 0.001349251124157869, 'topK': 633}. Best is trial 0 with value: 0.2885151505470276.
  Fold 1/5 - Score: 0.27499231696128845
  Fold 2/5 - Score: 0.27475881576538086
  Fold 3/5 - Score: 0.275919109582901


  Fold 4/5 - Score: 0.27513888478279114
[I 2025-11-21 13:24:53,448] Trial 16 finished with value: 0.27520227432250977 and parameters: {'l1_ratio': 0.6336150825705198, 'alpha': 0.00415211033411815, 'topK': 683}. Best is trial 0 with value: 0.2885151505470276.
  Fold 1/5 - Score: 0.28798606991767883
  Fold 2/5 - Score: 0.287773996591568
  Fold 3/5 - Score: 0.2887885272502899
  Fold 4/5 - Score: 0.28756481409072876


  Fold 5/5 - Score: 0.28947195410728455
[I 2025-11-21 13:32:34,593] Trial 17 finished with value: 0.28831708431243896 and parameters: {'l1_ratio': 0.37106981084546586, 'alpha': 0.0013725792372816987, 'topK': 664}. Best is trial 0 with value: 0.2885151505470276.
  Fold 1/5 - Score: 0.28762590885162354
  Fold 2/5 - Score: 0.2875833809375763
  Fold 3/5 - Score: 0.2887544333934784


  Fold 4/5 - Score: 0.2871771454811096
[I 2025-11-21 13:38:23,787] Trial 18 finished with value: 0.28778523206710815 and parameters: {'l1_ratio': 0.3588194046741692, 'alpha': 0.0015872289785684875, 'topK': 685}. Best is trial 0 with value: 0.2885151505470276.
  Fold 1/5 - Score: 0.28218236565589905
  Fold 2/5 - Score: 0.28219491243362427
  Fold 3/5 - Score: 0.2837907075881958


  Fold 4/5 - Score: 0.2816852331161499
[I 2025-11-21 13:42:50,138] Trial 19 finished with value: 0.28246331214904785 and parameters: {'l1_ratio': 0.38240042025465526, 'alpha': 0.0035984685575816983, 'topK': 633}. Best is trial 0 with value: 0.2885151505470276.


In [32]:
optuna.visualization.plot_optimization_history(optuna_study)

In [33]:
optuna.visualization.plot_param_importances(optuna_study)

In [34]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Trial 1:
Best Value: 0.28851595520973206
Best Params: {'l1_ratio': 0.42825676292409526, 'alpha': 0.0010061033871087777, 'topK': 670}

# **Train Model with best hyperparameter**

In [19]:
# Train final model on train + validation with best hyperparameters
URM_train, URM_validation = folds[0]

bp = optimizer.get_best_params(STUDY_NAME)

recommender = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train + URM_validation)
recommender.fit(
    l1_ratio=bp['l1_ratio'],
    alpha=bp['alpha'],
    positive_only=False,
    topK=bp['topK'],
    workers=8
)
# Save the trained model
recommender.save_model('tuning_slim')

100%|█████████▉| 6968/6969 [02:09<00:00, 53.62it/s]


SLIMElasticNetRecommender: Saving model in file 'tuning_slimSLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete


In [20]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, STUDY_NAME + "tuned_again" + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")